# 04 · Análise

O modelo dá o grau ICDR (0–4) e diz se há edema macular.

**Desfecho primário: grau ICDR.** Concordância entre o grau previsto e o de referência: kappa ponderado quadrático
(principal), kappa linear, acerto exato e erro de até 1 grau; matriz de confusão; comparação pareada entre modelos
(H1: MedGemma 4B vs. Gemma 3 4B) pela diferença de kappa. Respostas sem grau legível ficam fora e são contadas.

**Desfecho secundário: RD referenciável** = ICDR ≥ 2 e/ou edema macular (derivado da resposta; inválida = encaminhar).
Sensibilidade, especificidade, AUROC/AUPRC de P(referenciável) = 1 − P(grau 0 ou 1) × [1 − P(edema)] (coluna `p_yes`),
especificidade com sensibilidade fixa (limiar na metade A, avaliação na metade B), sensibilidade por grau, DeLong e McNemar.
Também: concordância entre prompts e entre execuções, linha de base supervisionada e eficiência.

Os dois olhos do paciente entram na análise: ICs por bootstrap por paciente, DeLong agrupado (Obuchowski),
McNemar agrupado (Durkalski) e metades A/B divididas por paciente.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import config
pd.set_option("display.max_columns", 50, "display.width", 250, "display.max_rows", 200)
from vlmdr import cohort, infer, metrics
from sklearn.metrics import cohen_kappa_score

In [ ]:
cohorts = {ds: cohort.load(ds).query("split == 'eval'") for ds in ("brset", "mbrset")}
p = infer.load_preds()
p = p[~p.run.astype(str).str.startswith("piloto")]
p["yhat"] = (p.answer != "no").astype(int)           # inválida = encaminhar
p["yhat_valid"] = p.answer.map({"yes": 1, "no": 0})  # análise de sensibilidade (exclui inválidas)
main = p[(p.prompt == "main_en") & (p.run == "r1")]
d = pd.concat([cohorts[ds].merge(g, on=["image_id", "patient_id", "dataset"]) for ds, g in main.groupby("dataset")])
d.groupby(["dataset", "model"]).size().unstack()

## Desfecho primário: grau ICDR (0–4)

In [ ]:
rows = []
for (ds, m), g in d.groupby(["dataset", "model"]):
    ok = g[g.grade_pred.notna()]
    yt, yp, pid = ok.dr_icdr.astype(int), ok.grade_pred.astype(int), ok.patient_id
    rows.append({"base": ds, "modelo": m, "n": len(g), "pacientes": g.patient_id.nunique(),
                 "sem grau legível": int(g.grade_pred.isna().sum()),
                 "kappa quadrático": metrics.fmt(*metrics.kappa(yt, yp, "quadratic", groups=pid), pct=False),
                 "kappa linear": metrics.fmt(*metrics.kappa(yt, yp, "linear", groups=pid), pct=False),
                 "acerto exato %": metrics.fmt(*metrics.proportion((yt == yp).astype(int), groups=pid)),
                 "erro de até 1 grau %": metrics.fmt(*metrics.proportion(((yt - yp).abs() <= 1).astype(int), groups=pid))})
grau = pd.DataFrame(rows)
config.TABLES.mkdir(parents=True, exist_ok=True)
grau.to_csv(config.TABLES / "primario_grau.csv", index=False)
grau

### Matriz de confusão (linhas = grau real; colunas = previsto; -1 = sem grau legível) e % de acerto por grau real

In [ ]:
for (ds, m), g in d.groupby(["dataset", "model"]):
    print(ds, m)
    cm = pd.crosstab(g.dr_icdr.rename("real"), g.grade_pred.fillna(-1).astype(int).rename("previsto"))
    cm["acerto %"] = [round(100 * cm.loc[k, k] / cm.loc[k].sum(), 1) if k in cm.columns else 0.0 for k in cm.index]
    display(cm)

### Comparações pareadas do grau (diferença de kappa quadrático; bootstrap pareado por paciente)

In [ ]:
modelos = sorted(d.model.unique())
pares = [(x, y) for i, x in enumerate(modelos) for y in modelos[i + 1:]]
kq = lambda yt, yp: cohen_kappa_score(yt, yp, weights="quadratic", labels=metrics.ICDR)

def pareado_grau(ds, a, b):
    w = d[d.dataset == ds].pivot_table(index=["image_id", "patient_id", "dr_icdr"], columns="model",
                                       values="grade_pred").dropna()
    y = w.index.get_level_values("dr_icdr").values.astype(int)
    pid = w.index.get_level_values("patient_id").values
    est, lo, hi, p = metrics.paired_diff(kq, y, w[a].values.astype(int), w[b].values.astype(int), groups=pid)
    return {"base": ds, "A": a, "B": b, "n": len(w), "pacientes": len(set(pid)),
            "kappa A": round(kq(y, w[a].values.astype(int)), 3), "kappa B": round(kq(y, w[b].values.astype(int)), 3),
            "diferença (IC 95%)": metrics.fmt(est, lo, hi, pct=False), "p": p}

comp_grau = pd.DataFrame([pareado_grau(ds, a, b) for ds in d.dataset.unique() for a, b in pares])
comp_grau.to_csv(config.TABLES / "primario_grau_pareado.csv", index=False)
comp_grau

## Desfecho secundário: RD referenciável

In [ ]:
rows = []
for (ds, m), g in d.groupby(["dataset", "model"]):
    r = metrics.sens_spec(g.referable, g.yhat, groups=g.patient_id)
    auc = metrics.auroc(g.referable, g.p_yes, groups=g.patient_id)
    rows.append({"base": ds, "modelo": m, "n": r["n"], "pacientes": r["pacientes"], "positivos": r["positivos"],
                 "sensibilidade %": metrics.fmt(r["sens"], r["sens_lo"], r["sens_hi"]),
                 "especificidade %": metrics.fmt(r["spec"], r["spec_lo"], r["spec_hi"]),
                 "AUROC": metrics.fmt(*auc, pct=False),
                 "inválidas": int((g.answer == "invalid").sum())})
sec = pd.DataFrame(rows)
sec.to_csv(config.TABLES / "secundario_referenciavel.csv", index=False)
sec

## Comparações pareadas do referenciável dentro de cada base (McNemar e DeLong agrupados)

In [ ]:
def pareado(ds, a, b):
    w = d[d.dataset == ds].pivot_table(index=["image_id", "patient_id", "referable"], columns="model",
                                       values=["yhat", "p_yes"]).dropna()
    y = w.index.get_level_values("referable").values
    pid = w.index.get_level_values("patient_id").values
    pos, neg = y == 1, y == 0
    p_sens = metrics.mcnemar(w["yhat"][a].values[pos] == 1, w["yhat"][b].values[pos] == 1, groups=pid[pos])
    p_spec = metrics.mcnemar(w["yhat"][a].values[neg] == 0, w["yhat"][b].values[neg] == 0, groups=pid[neg])
    a1, a2, diff, p_auc = metrics.delong(y, w["p_yes"][a].values, w["p_yes"][b].values, groups=pid)
    return {"base": ds, "A": a, "B": b, "n": len(w), "pacientes": len(set(pid)),
            "p McNemar sens": p_sens, "p McNemar espec": p_spec,
            "AUROC A": a1, "AUROC B": a2, "p DeLong": p_auc}

comp = pd.DataFrame([pareado(ds, a, b) for ds in d.dataset.unique() for a, b in pares])
comp

## Ponto de operação do rastreio e triagem (limiar fixado na metade A, avaliado na metade B)

In [ ]:
rows = []
for (ds, m), g in d.groupby(["dataset", "model"]):
    for alvo in (0.90, 0.95):
        r = metrics.operating_point(g.referable, g.p_yes, alvo, groups=g.patient_id)
        prev = g.referable.mean()
        ppv, npv = metrics.predictive_values(r["sens"], r["spec"], prev)
        rows.append({"base": ds, "modelo": m, "alvo": alvo,
                     "sens %": metrics.fmt(r["sens"], r["sens_lo"], r["sens_hi"]),
                     "espec %": metrics.fmt(r["spec"], r["spec_lo"], r["spec_hi"]),
                     "VPP": round(ppv, 3), "VPN": round(npv, 3),
                     "sem leitura %": round(r["triagem_sem_leitura_%"], 1), "perdidos/1000": round(r["perdidos_por_1000"], 1)})
pd.DataFrame(rows)

## Edema macular

In [ ]:
rows = []
for (ds, m), g in d.groupby(["dataset", "model"]):
    e = g.edema_pred.notna()
    r = metrics.sens_spec(g.macular_edema[e], (g.edema_pred[e] == "yes").astype(int), groups=g.patient_id[e])
    rows.append({"base": ds, "modelo": m, "edema legível": int(e.sum()), "com edema (real)": r["positivos"],
                 "sensibilidade %": metrics.fmt(r["sens"], r["sens_lo"], r["sens_hi"]),
                 "especificidade %": metrics.fmt(r["spec"], r["spec_lo"], r["spec_hi"]),
                 "kappa": round(cohen_kappa_score(g.macular_edema[e], (g.edema_pred[e] == "yes").astype(int)), 3)})
pd.DataFrame(rows)

## Sensibilidade para referenciável por grau real

In [ ]:
pd.concat([metrics.sens_by_grade(g).assign(base=ds, modelo=m) for (ds, m), g in d.groupby(["dataset", "model"])])

## Robustez ao prompt e reprodutibilidade

In [ ]:
def concordancia(p1, r1, p2, r2):
    cols = ["model", "dataset", "image_id", "grade_pred", "answer"]
    a = p[(p.prompt == p1) & (p.run == r1)][cols]
    b = p[(p.prompt == p2) & (p.run == r2)][cols]
    m = a.merge(b, on=["model", "dataset", "image_id"], suffixes=("_a", "_b"))
    out = []
    for (mod, ds), g in m.groupby(["model", "dataset"]):
        gg = g.dropna(subset=["grade_pred_a", "grade_pred_b"])
        out.append({"comparação": f"{p1}/{r1} vs {p2}/{r2}", "modelo": mod, "base": ds, "n": len(g),
                    "grau igual %": round(100 * (gg.grade_pred_a == gg.grade_pred_b).mean(), 1),
                    "kappa quadrático (grau)": round(cohen_kappa_score(gg.grade_pred_a.astype(int), gg.grade_pred_b.astype(int),
                                                                       weights="quadratic", labels=metrics.ICDR), 3),
                    "referenciável igual %": round(100 * (g.answer_a == g.answer_b).mean(), 1),
                    "kappa (referenciável)": round(cohen_kappa_score(g.answer_a, g.answer_b), 3)})
    return out

pd.DataFrame(concordancia("main_en", "r1", "nodef_en", "r1") + concordancia("main_en", "r1", "main_pt", "r1")
             + concordancia("main_en", "r1", "main_en", "r2"))

## Linha de base supervisionada (escolhida no notebook 01; só o desfecho secundário, referenciável)

In [ ]:
EMBEDDING = "dinov3_convnext_tiny"   # troque pela escolha registrada
bl = pd.read_csv(config.PREDS / f"baseline_{EMBEDDING}.csv", dtype={"image_id": str})
rows = []
for (setting, ds), g in bl.groupby(["setting", "dataset"]):
    c = cohorts[ds].merge(g, on="image_id")
    rows.append({"cenário": setting, "base": ds, "n": len(c), "pacientes": c.patient_id.nunique(),
                 "AUROC": metrics.fmt(*metrics.auroc(c.referable, c.p, groups=c.patient_id), pct=False)})
pd.DataFrame(rows)

## Eficiência (tempo, tokens)

In [ ]:
main.groupby(["model", "dataset"]).agg(seg_mediana=("seconds", "median"), tokens_in=("n_tokens_in", "median"),
                                      tokens_out=("n_tokens_out", "median"), dispositivo=("device", "first"))